# Actividad Extra: Analisis de Datos de Vuelos

## Objetivos
- Leer y explorar un dataset de vuelos
- Crear columnas derivadas a partir de datos existentes
- Aplicar pivoting para analisis multidimensional
- Crear funciones personalizadas de agregacion
- Construir un pipeline de Machine Learning con Spark MLlib

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("actividad_vuelos") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

---
## Ejercicio 1: Lectura y exploracion del dataset

Lee el archivo `flights.csv`, muestra el schema y las primeras 5 filas, y cuenta el total de registros.

**Columnas del dataset:** year, month, day, dep_time, dep_delay, arr_time, arr_delay, carrier, tailnum, flight, origin, dest, air_time, distance, hour, minute

In [ ]:
# =============================================================
# EJERCICIO 1: Lectura y exploracion
# =============================================================
# TODO: Lee el archivo /home/jovyan/datos/flights.csv con header=True e inferSchema=True
# TODO: Muestra el schema con printSchema()
# TODO: Muestra las primeras 5 filas con show(5)
# TODO: Cuenta el total de registros con count()
#
# Pista: spark.read.csv(path, header=True, inferSchema=True)

# Escribe tu codigo aqui:


---
## Ejercicio 2: Crear columna de fecha

Combina las columnas `year`, `month` y `day` para crear una nueva columna `fecha` de tipo Date.

**Pista:** Usa `F.to_date(F.concat_ws("-", "year", "month", "day"))` para concatenar y convertir a fecha.

In [ ]:
# =============================================================
# EJERCICIO 2: Crear columna de fecha
# =============================================================
# TODO: Usa withColumn para crear la columna "fecha" combinando year, month, day
# TODO: Muestra las columnas year, month, day, fecha para verificar
#
# Pista: df = df.withColumn("fecha", F.to_date(F.concat_ws("-", "year", "month", "day")))

# Escribe tu codigo aqui:


---
## Ejercicio 3: Dia de la semana

A partir de la columna `fecha`, crea una columna `dia_semana` con el numero de dia (1=Domingo...7=Sabado) y luego mapea a nombre legible.

**Pista:** Usa `F.dayofweek("fecha")` y luego `F.when()` encadenado para mapear numeros a nombres.

In [ ]:
# =============================================================
# EJERCICIO 3: Dia de la semana
# =============================================================
# TODO: Crea la columna "dia_semana" con F.dayofweek("fecha")
# TODO: Crea la columna "dia_nombre" mapeando:
#   1 -> "Domingo", 2 -> "Lunes", 3 -> "Martes", 4 -> "Miercoles",
#   5 -> "Jueves", 6 -> "Viernes", 7 -> "Sabado"
# TODO: Muestra una muestra de las columnas fecha, dia_semana, dia_nombre
#
# Pista: F.when(F.col("dia_semana") == 1, "Domingo").when(...).otherwise(...)

# Escribe tu codigo aqui:


---
## Ejercicio 4: Pivoting - Vuelos por dia de semana y aerolinea

Crea una tabla pivoteada que muestre la cantidad de vuelos por aerolinea (`carrier`) para cada dia de la semana.

**Pista:** Usa `.groupBy("carrier").pivot("dia_nombre").count()`

In [ ]:
# =============================================================
# EJERCICIO 4: Pivoting
# =============================================================
# TODO: Agrupa por "carrier", pivotea por "dia_nombre" y cuenta los vuelos
# TODO: Muestra el resultado completo con show(truncate=False)
#
# Pista: df.groupBy("carrier").pivot("dia_nombre").count().show(truncate=False)

# Escribe tu codigo aqui:


---
## Ejercicio 5: Retrasos medios por periodo del dia

Crea una funcion que clasifique los vuelos segun el periodo del dia basandose en la columna `hour`:
- 06-11: Manana
- 12-17: Tarde
- 18-23: Noche
- 00-05: Madrugada

Luego calcula el promedio de `dep_delay` por cada periodo.

In [ ]:
# =============================================================
# EJERCICIO 5: Retrasos medios por periodo del dia
# =============================================================
# TODO: Crea una columna "periodo" usando F.when() basandote en la columna "hour":
#   - hour entre 6 y 11 -> "Manana"
#   - hour entre 12 y 17 -> "Tarde"
#   - hour entre 18 y 23 -> "Noche"
#   - hour entre 0 y 5 -> "Madrugada"
# TODO: Agrupa por "periodo" y calcula el promedio de dep_delay
# TODO: Ordena por retraso promedio descendente y muestra
#
# Pista: F.when((F.col("hour") >= 6) & (F.col("hour") <= 11), "Manana")
#        .when(...).otherwise(...)

# Escribe tu codigo aqui:


---
## Ejercicio 6: Pipeline de Machine Learning

Construye un pipeline de ML para predecir si un vuelo tendra un retraso significativo (>15 min).

**Pasos:**
1. Usa `StringIndexer` para codificar `carrier`
2. Usa `VectorAssembler` para combinar features: carrier_idx, month, distance, hour
3. Usa `Binarizer` para crear la variable objetivo: dep_delay > 15 -> 1, sino -> 0
4. Entrena un `DecisionTreeClassifier`
5. Evalua la precision (accuracy)

In [ ]:
# =============================================================
# EJERCICIO 6: Pipeline de Machine Learning
# =============================================================
# TODO: Importa las clases necesarias:
#   from pyspark.ml.feature import StringIndexer, VectorAssembler, Binarizer
#   from pyspark.ml.classification import DecisionTreeClassifier
#   from pyspark.ml.evaluation import MulticlassClassificationEvaluator
#   from pyspark.ml import Pipeline
#
# TODO: Elimina filas con nulls en dep_delay
# TODO: Crea la columna "label" con Binarizer (threshold=15, inputCol="dep_delay")
#   o bien con F.when(F.col("dep_delay") > 15, 1.0).otherwise(0.0)
# TODO: Crea StringIndexer para "carrier" -> "carrier_idx"
# TODO: Crea VectorAssembler con inputCols=["carrier_idx", "month", "distance", "hour"]
# TODO: Divide en train/test (70/30)
# TODO: Entrena DecisionTreeClassifier
# TODO: Evalua accuracy con MulticlassClassificationEvaluator

# Escribe tu codigo aqui:


---
## Resumen

En esta actividad aprendimos:

1. **Lectura y exploracion** de un dataset de vuelos
2. **Creacion de columnas derivadas** con `withColumn`, `to_date`, `concat_ws`
3. **Mapeo de valores** con `F.when()` encadenado
4. **Pivoting** con `groupBy().pivot().count()`
5. **Funciones de agregacion** personalizadas por periodo
6. **Pipeline ML** con StringIndexer, VectorAssembler y DecisionTreeClassifier

In [ ]:
# Detener SparkSession
spark.stop()
print("SparkSession detenida correctamente.")